# CircuitLM - Fast Training (sentencepiece BPE, &le;30 min)

Trains a **hybrid CircuitLM** (integer FSM circuit + neural corrector) with **1024-token BPE vocab**.

**Key changes vs old notebook:**
- Tokenizer: pure-Python BPE (slow) → sentencepiece C++ (1-2 sec)
- Circuit: CP-SAT joint PDA (5+ hrs) → Greedy FSM (seconds)

**Data:** `/kaggle/input/datasets/zacharymaronek/circuit-lm-personal/all_personal_training.txt`

**Runtime:** GPU required. P100 or T4. ~10-20 min total.

## 1. Clone CircuitLM + Install Dependencies

In [1]:
%pip install "numpy<2" ortools msgpack sentencepiece torch --quiet

!git clone https://github.com/toxzak-svg/circuit_lm.git
!cd circuit_lm && git checkout origin/master

import sys
sys.path.insert(0, 'circuit_lm/src')
print('Setup complete')

Note: you may need to restart the kernel to use updated packages.
fatal: destination path 'circuit_lm' already exists and is not an empty directory.
error: Your local changes to the following files would be overwritten by checkout:
	kaggle_training_notebook.ipynb
Please commit your changes or stash them before you switch branches.
Aborting
Setup complete


## 2. Load Training Data

In [2]:
import os

DATA_PATH = 'all_personal_training.txt'

with open(DATA_PATH, encoding='utf-8', errors='replace') as f:
    text = f.read()

lines = [l for l in text.strip().split('\n') if l.strip()]
print(f'Loaded {len(lines)} lines ({len(text)/1024/1024:.1f} MB)')

Loaded 24414 lines (6.9 MB)


## 3. Tokenizer — 1024 BPE Vocab via sentencepiece (fast!)

sentencepiece builds 1024-token BPE vocab in ~1-2 seconds on 7MB of text.
Saved as circuit_lm JSON format so the rest of the pipeline works unchanged.

In [3]:
import json
import os
import time
import sentencepiece as spm

import sys
sys.path.insert(0, 'circuit_lm/src')
from circuit_lm.tokenizer import Tokenizer

VOCAB_SIZE = 1024
TOKENIZER_JSON = '/tmp/tokenizer_1k.json'
SP_MODEL = '/tmp/sp_bpe_1k.model'

if os.path.exists(TOKENIZER_JSON):
    print(f'Loading cached tokenizer from {TOKENIZER_JSON}...')
    with open(TOKENIZER_JSON) as f:
        data = json.load(f)
    tokenizer = Tokenizer.from_dict(data)
    print(f'Loaded vocab_size={tokenizer.vocab_size}')
else:
    print(f'Building {VOCAB_SIZE}-token BPE vocab with sentencepiece (~1-2 sec)...')
    t0 = time.time()
    spm.SentencePieceTrainer.train(
        input=DATA_PATH,
        model_prefix='/tmp/sp_bpe_1k',
        vocab_size=VOCAB_SIZE,
        character_coverage=0.9995,
        model_type='bpe',
        pad_id=0,
        unk_id=1,
        bos_id=-1,
        eos_id=-1,
        minloglevel=1,
    )
    print(f'Built in {time.time()-t0:.1f}s')
    
    # Load and convert to circuit_lm Tokenizer format
    sp = spm.SentencePieceProcessor()
    sp.load(SP_MODEL)
    pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]
    data = {'mode': 'bpe', 'pieces': pieces}
    with open(TOKENIZER_JSON, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    tokenizer = Tokenizer.from_dict(data)
    print(f'Saved tokenizer JSON to {TOKENIZER_JSON}')
    print(f'circuit_lm tokenizer vocab_size={tokenizer.vocab_size}')

from circuit_lm.data import load_sequences
sequences = load_sequences(DATA_PATH, tokenizer)
print(f'Tokenization complete. {len(sequences)} sequences, avg len={sum(len(s) for s in sequences)/len(sequences):.0f}')

Loading cached tokenizer from /tmp/tokenizer_1k.json...
Loaded vocab_size=1024
Tokenization complete. 17464 sequences, avg len=257


## 4. Train FSM Circuit — Greedy (NOT CP-SAT)

Uses `train_greedy.train_fast` — O(S×V) greedy argmax, finishes in seconds.
Refinement rounds = 2 for EM-like improvement.

In [4]:
from circuit_lm.train_greedy import train_fast
from circuit_lm.io import save_model
import time

STATE_BITS = 6   # 64 states
CONTEXT_LEN = 4  # tokens of context for state hashing

print(f'Training FSM circuit: {STATE_BITS} bits ({2**STATE_BITS} states), context_len={CONTEXT_LEN}')
t0 = time.time()

circuit = train_fast(
    sequences=sequences,
    vocab_size=tokenizer.vocab_size,
    state_bits=STATE_BITS,
    steps=0,
    context_len=CONTEXT_LEN,
    top_k_coverage=16,
    refinement_rounds=2,
)

circuit_time = time.time() - t0
print(f'Circuit trained in {circuit_time:.1f}s')

CIRCUIT_PATH = '/tmp/circuit_fsm_1k.json'
save_model(circuit, tokenizer, CIRCUIT_PATH)
print(f'Circuit saved: {os.path.getsize(CIRCUIT_PATH)/1024:.0f} KB')

Training FSM circuit: 6 bits (64 states), context_len=4
Circuit trained in 11.6s
Circuit saved: 1759 KB


## 5. Train 4 Corrector Sizes (GPU)

All 4 share the same circuit. Only the neural corrector changes.

| Size    | embed_dim | hidden_dim | num_layers | context | Params     |
|---------|-----------|------------|------------|---------|------------|
| Tiny    | 64        | 128        | 2          | 32      | ~1.5 M     |
| Small   | 128       | 256        | 3          | 64      | ~5 M       |
| Medium  | 256       | 512        | 4          | 64      | ~20 M      |
| Large   | 512       | 1024       | 4          | 128     | ~80 M      |

P100 (16GB) handles Medium comfortably. Use T4 if only 15GB.

In [5]:
import torch
import time
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: NVIDIA RTX A4000
VRAM: 15.7 GB


In [6]:
# ---- Corrector Configs ----
CORRECTOR_CONFIGS = {
    'tiny':   dict(embed_dim=64,   hidden_dim=128,  num_layers=2, max_context_len=32),
    'small':  dict(embed_dim=128,  hidden_dim=256,  num_layers=3, max_context_len=64),
    'medium': dict(embed_dim=256,  hidden_dim=512,  num_layers=4, max_context_len=64),
    'large':  dict(embed_dim=512,  hidden_dim=1024, num_layers=4, max_context_len=128),
}

TRAINING_BUDGETS = {
    'tiny':   dict(num_epochs=5,  batch_size=256, max_examples=80000),
    'small':  dict(num_epochs=5,  batch_size=128, max_examples=80000),
    'medium': dict(num_epochs=5,  batch_size=64,  max_examples=60000),
    'large':  dict(num_epochs=3,  batch_size=32,  max_examples=40000),
}

SIZES_TO_TRAIN = ['large']

print('Corrector sizes to train:', SIZES_TO_TRAIN)

Corrector sizes to train: ['large']


In [7]:
import sys
sys.path.insert(0, 'src')
from hybrid import train_hybrid, HybridModel
from circuit_lm.io import load_model

RESULTS = {}

for size in SIZES_TO_TRAIN:
    print('\n' + '='*50)
    print(f'TRAINING CORRECTOR: {size.upper()}')
    print('='*50)
    
    cfg = CORRECTOR_CONFIGS[size]
    tcfg = TRAINING_BUDGETS[size]
    
    print(f'  embed_dim={cfg["embed_dim"]}, hidden_dim={cfg["hidden_dim"]}, '
          f'num_layers={cfg["num_layers"]}, context_len={cfg["max_context_len"]}')
    print(f'  epochs={tcfg["num_epochs"]}, batch_size={tcfg["batch_size"]}, '
          f'max_examples={tcfg["max_examples"]}')
    
    t0 = time.time()
    try:
        corrector = train_hybrid(
            circuit_path=CIRCUIT_PATH,
            data_path=DATA_PATH,
            output_path=f'/tmp/corrector_{size}_1k.pt',
            num_epochs=tcfg['num_epochs'],
            batch_size=tcfg['batch_size'],
            circuit_weight=0.5,
            max_examples=tcfg['max_examples'],
            embed_dim=cfg['embed_dim'],
            hidden_dim=cfg['hidden_dim'],
            num_layers=cfg['num_layers'],
            max_context_len=cfg['max_context_len'],
            use_ssd=True,
        )
        elapsed = time.time() - t0
        num_params = sum(p.numel() for p in corrector.corrector.parameters())
        
        # Quick sample test
        hybrid, tok = HybridModel.load(CIRCUIT_PATH, f'/tmp/corrector_{size}_1k.pt')
        hybrid.corrector.to(device)
        hybrid.corrector.eval()
        prompt_ids = tok.encode('The best way to build AI that you can trust is')
        from src.hybrid import generate_reply_hybrid
        reply_ids = generate_reply_hybrid(hybrid, tok, prompt_ids, max_tokens=40)
        reply = tok.decode(reply_ids)
        
        RESULTS[size] = {
            'status': 'success',
            'time': elapsed,
            'params': num_params,
            'sample': reply,
            'config': {**cfg, **tcfg},
        }
        print(f'  OK - {size} done in {elapsed:.1f}s - {num_params:,} params')
        print(f'  Sample: {reply[:120]}...')
        
        del corrector, hybrid
        torch.cuda.empty_cache()
        
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM on {size} — reduce batch_size or max_examples')
            RESULTS[size] = {'status': 'OOM', 'config': {**cfg, **tcfg}}
            torch.cuda.empty_cache()
        else:
            raise


TRAINING CORRECTOR: LARGE
  embed_dim=512, hidden_dim=1024, num_layers=4, context_len=128
  epochs=3, batch_size=32, max_examples=40000
Loading circuit from /tmp/circuit_fsm_1k.json...
Building dataset from all_personal_training.txt...
Built 40000 training examples
CircuitLM accuracy: 26.12%
NeuralCorrector (SSD): 9,218,048 parameters
Training on cuda
Epoch 1/3: loss=4.2391, accuracy=26.32%
Epoch 2/3: loss=3.9441, accuracy=27.37%
Epoch 3/3: loss=3.7986, accuracy=28.08%
Saved to /tmp/corrector_large_1k.pt


RuntimeError: Error(s) in loading state_dict for NeuralCorrector:
	Unexpected key(s) in state_dict: "mlp.9.weight", "mlp.9.bias", "mlp.12.weight", "mlp.12.bias". 
	size mismatch for state_embed.weight: copying a param with shape torch.Size([65, 512]) from checkpoint, the shape in current model is torch.Size([65, 64]).
	size mismatch for stack_embed.weight: copying a param with shape torch.Size([5, 512]) from checkpoint, the shape in current model is torch.Size([5, 64]).
	size mismatch for token_embed.weight: copying a param with shape torch.Size([1024, 512]) from checkpoint, the shape in current model is torch.Size([1024, 64]).
	size mismatch for circuit_proj.0.weight: copying a param with shape torch.Size([1024, 1024]) from checkpoint, the shape in current model is torch.Size([128, 1024]).
	size mismatch for circuit_proj.0.bias: copying a param with shape torch.Size([1024]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for circuit_proj.2.weight: copy

## 6. Results Summary

In [ ]:
print('\n' + '='*60)
print('TRAINING RESULTS')
print('='*60)
print(f'Vocabulary: {VOCAB_SIZE} tokens (sentencepiece BPE, built in ~1s)')
print(f'Circuit: FSM greedy, {STATE_BITS} bits ({2**STATE_BITS} states), context_len={CONTEXT_LEN}')
print()

for size in SIZES_TO_TRAIN:
    r = RESULTS.get(size, {})
    status = r.get('status', 'not_run')
    if status == 'success':
        cfg = r['config']
        print(f'{size.upper():8} | {r["params"]:>8,} params | {r["time"]:>6.1f}s | '
              f'embed={cfg["embed_dim"]:4d} hidden={cfg["hidden_dim"]:4d} layers={cfg["num_layers"]}')
    elif status == 'OOM':
        print(f'{size.upper():8} | OOM — reduce batch_size or max_examples')
    else:
        print(f'{size.upper():8} | {status}')

## 7. Download All Files

In [ ]:
from IPython.display import FileLink, display

print('='*60)
print('DOWNLOAD YOUR TRAINED FILES')
print('='*60)
print()

print('CIRCUIT (shared by all corrector sizes):')
display(FileLink(CIRCUIT_PATH))
print()

print('TOKENIZER (circuit_lm JSON format):')
display(FileLink(TOKENIZER_JSON))
print()

print('SENTENCEPIECE MODEL (for inference):')
display(FileLink(SP_MODEL))
print()

print('CORRECTOR MODELS:')
for size in SIZES_TO_TRAIN:
    if RESULTS.get(size, {}).get('status') == 'success':
        path = f'/tmp/corrector_{size}_1k.pt'
        r = RESULTS[size]
        print(f'  [{size.upper()}] {r["params"]:,} params, {r["time"]:.1f}s —')
        display(FileLink(path))
        print()

tokenizer_info = {
    'vocab_size': tokenizer.vocab_size,
    'mode': 'bpe',
}
with open('/tmp/tokenizer_info.json', 'w') as f:
    json.dump(tokenizer_info, f)
print('Tokenizer config:')
display(FileLink('/tmp/tokenizer_info.json'))

total_time = sum(r.get('time', 0) for r in RESULTS.values() if r.get('status') == 'success')
print(f'\nCircuit time: {circuit_time:.1f}s')
print(f'Total GPU training time: {total_time:.1f}s')

## 8. Compare All Corrector Sizes

In [ ]:
TEST_PROMPT = 'The best way to build AI that you can trust is'
MAX_TOKENS = 60

print(f'Prompt: {TEST_PROMPT}')
print('-'*60)

for size in SIZES_TO_TRAIN:
    if RESULTS.get(size, {}).get('status') != 'success':
        continue
    
    hybrid, tok = HybridModel.load(CIRCUIT_PATH, f'/tmp/corrector_{size}_1k.pt')
    hybrid.corrector.to(device)
    hybrid.corrector.eval()
    
    prompt_ids = tok.encode(TEST_PROMPT)
    reply_ids = generate_reply_hybrid(hybrid, tok, prompt_ids, max_tokens=MAX_TOKENS)
    reply = tok.decode(reply_ids)
    
    r = RESULTS[size]
    print(f'[{size.upper():8} {r["params"]:>8,} params] {reply}')
    print()
    
    del hybrid
    torch.cuda.empty_cache()

## Notes

**What changed vs the old notebook:**
- Vocab: pure-Python BPE → sentencepiece C++ (~1-2 sec vs 5+ min)
- Circuit: CP-SAT joint PDA (5+ hours) → Greedy FSM (seconds)
- Stack: PDA with push/pop → simple hash-FSM (no stack, corrector handles the rest)

**Why this works:** The neural corrector is doing the real heavy lifting. A greedy FSM circuit gives it a reasonable starting point — the corrector fixes the rest. No need to prove mathematical optimality when you have a 20M-param neural net backing you up.

**Scaling:** Medium (20M params) is the sweet spot for P100. Large is there if you want to push quality.

**Next steps:**
1. Download `circuit_fsm_1k.json` + `tokenizer_1k.json` + `sp_bpe_1k.model` + the corrector you want
2. Wire into the Rust inference kernel for GPU-free local inference
3. Use evolver to search for better training recipes